In [1]:
import numpy as np
from sklearn.model_selection import train_test_split
import pandas as pd 

train = pd.read_csv('../../data/house-prices/train.csv')
test = pd.read_csv('../../data/house-prices/test.csv')

# Find columns where more than 50% of rows are NaN
limit = len(train) * 0.5
cols_to_drop = train.columns[train.isna().sum() > limit]

print(f"Dropped columns: {list(cols_to_drop)}")


train.drop(cols_to_drop, axis=1, inplace=True)

X = train.drop("SalePrice", axis=1)
y = train["SalePrice"]

# This gives you a y_test so you can actually check your accuracy/error
train_data, val_data, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

Dropped columns: ['Alley', 'MasVnrType', 'PoolQC', 'Fence', 'MiscFeature']


In [61]:
train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1458 entries, 0 to 1459
Data columns (total 76 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1458 non-null   int64  
 1   MSSubClass     1458 non-null   int64  
 2   MSZoning       1458 non-null   object 
 3   LotFrontage    1199 non-null   float64
 4   LotArea        1458 non-null   int64  
 5   Street         1458 non-null   object 
 6   LotShape       1458 non-null   object 
 7   LandContour    1458 non-null   object 
 8   Utilities      1458 non-null   object 
 9   LotConfig      1458 non-null   object 
 10  LandSlope      1458 non-null   object 
 11  Neighborhood   1458 non-null   object 
 12  Condition1     1458 non-null   object 
 13  Condition2     1458 non-null   object 
 14  BldgType       1458 non-null   object 
 15  HouseStyle     1458 non-null   object 
 16  OverallQual    1458 non-null   int64  
 17  OverallCond    1458 non-null   int64  
 18  YearBuilt    

In [49]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder




num_attribs = [name for name in train_data.columns.tolist() if train_data[name].dtype == "int64" or train[name].dtype == "float64" ]
cat_attribs = [name for name in train_data.columns.tolist() if train_data[name].dtype == "object" ]

num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])


cat_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("cat_encoder", OneHotEncoder(handle_unknown='ignore')),
    ])



In [51]:
from sklearn.compose import ColumnTransformer


preprocess_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("cat", cat_pipeline, cat_attribs),
    ])

X_train = preprocess_pipeline.fit_transform(
    train_data[num_attribs + cat_attribs])


In [64]:
from sklearn.ensemble import RandomForestRegressor

y_train_log = np.log1p(y_train)

forest_clf = RandomForestRegressor(n_estimators=100, random_state=42)
forest_clf.fit(X_train, y_train_log)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [65]:
X_val = preprocess_pipeline.transform(val_data[num_attribs + cat_attribs])
y_pred_log = forest_clf.predict(X_val)
y_pred = np.expm1(y_pred_log)


In [67]:
from sklearn.metrics import mean_squared_error
mse = mean_squared_error(y_val, y_pred)

# 2. Calculate RMSE (This is usually better because it's in "Dollars")
rmse = np.sqrt(mse)

print(f"The average error is: ${rmse:,.2f}")

The average error is: $29,532.21


In [68]:
from sklearn.model_selection import cross_val_score

forest_scores = cross_val_score(forest_clf, X_train, y_train, cv=10)
forest_scores.mean()

np.float64(0.8394745072000065)